In [47]:
## RAGAS is a framework for evaluating the RAG pipeline.

## Metrics:
# 1. Context Relevancy: Measures how relevant the retrieved context is to the question asked.
#    It filters out irrelevant information from the retrieved context.
#    Calculated as: number of relevant sentences in context / total sentences in context.

# 2. Context Precision: Measures the signal-to-noise ratio of the retrieved context.
#    It evaluates whether the relevant chunks are ranked higher than irrelevant ones.
#    Calculated as: mean of precision@k for each relevant chunk in the ranked retrieved context,
#    where precision@k = number of relevant chunks in top-k / k.

# 3. Context Recall: Measures how much of the ground truth is captured in the retrieved context.
#    It checks if all the necessary information to answer the question was retrieved.
#    Calculated as: Recall@K = number of ground truth sentences attributable to context / total sentences in ground truth.

# 4. Faithfulness: Measures how factually consistent the generated answer is with the retrieved context.
#    It ensures the answer does not contain information not present in the context (no hallucinations).
#    Calculated as: number of answer statements that can be inferred from context / total statements in answer.

# 5. Answer Relevancy: Measures how relevant the generated answer is to the original question.
#    It penalizes answers that are incomplete or contain redundant information.
#    Calculated as: mean cosine similarity between the original question and n questions
#    generated from the answer, using embeddings to capture semantic similarity.

In [48]:
import nest_asyncio
nest_asyncio.apply()  # Allow asyncio to run inside Jupyter's existing event loop (needed by RAGAS)

import ragas
print(ragas.__version__)

import os
import pandas as pd
from dotenv import load_dotenv

# LangChain components
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI

# RAGAS wrappers to make LangChain LLM/Embeddings compatible with RAGAS
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Euriai embeddings (used for vector similarity)
from euriai.langchain import EuriaiEmbeddings

# Load environment variables from app/.env file
app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))


0.4.3


True

In [55]:
api_key = os.getenv("key")

# ── Option 1 (Active): OpenRouter free model via ChatOpenAI ───────────────────
# Good for testing — no cost, but may have rate limits and weaker reasoning.
# llm = ChatOpenAI(
#     model="stepfun/step-3.5-flash:free",
#     base_url="https://openrouter.ai/api/v1",
#     api_key="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",
# )

# ── Option 2 (Alternative): Euriai GPT-4o-mini ────────────────────────────────
# Better quality but requires Euriai API key. Uncomment below to use instead.
from euriai.langchain import create_chat_model
llm = create_chat_model(api_key=api_key, model="gpt-4o-mini", temperature=0.7)

# Single alias so all downstream cells reference the same model
model = llm

# Quick sanity-check — verify the LLM responds correctly before running RAGAS
response = llm.invoke("What is the capital of France?")
print("LLM test response:", response.content)

# ── Embeddings: Euriai text-embedding-3-small ─────────────────────────────────
# Used for both vector store (Chroma) and RAGAS answer relevancy metric
embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)


LLM test response: The capital of France is Paris.


In [56]:
# Load all .txt files from the data/ directory as LangChain Documents
loader = DirectoryLoader("./data", glob="**/*.txt")
docs = loader.load()

# Split documents into smaller chunks for the vector store
# chunk_size=350 tokens, chunk_overlap=20 for context continuity
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(docs)

print(f"Loaded {len(docs)} documents → split into {len(chunks)} chunks")


Loaded 3 documents → split into 33 chunks


In [57]:
docs

[Document(metadata={'source': 'data\\food.txt'}, page_content='margherita pizza; $12; classic with tomato, mozzarella, and basil; main dish\n\nspaghetti carbonara; $15; creamy pasta with pancetta and parmesan; main dish\n\nbruschetta; $8; toasted bread with tomato, garlic, and olive oil; appetizer\n\ncaprese salad; $10; fresh tomatoes, mozzarella, and basil; salad\n\nlasagna; $14; layered pasta with meat sauce and cheese; main dish\n\ntiramisu; $9; coffee-flavored italian dessert; dessert\n\ngelato; $7; traditional italian ice cream; dessert\n\nrisotto milanese; $16; creamy saffron-infused rice dish; main dish\n\npolenta; $11; cornmeal dish, often served as a side; side dish\n\nosso buco; $20; braised veal shanks with vegetables and broth; main dish\n\nravioli; $13; stuffed pasta with cheese or meat filling; main dish\n\nminestrone soup; $9; vegetable soup with pasta or rice; soup\n\nprosecco; $8; italian sparkling white wine; drink\n\nchianti; $10; dry red wine from tuscany; drink\n\n

In [58]:
import threading
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset.synthesizers import default_query_distribution

# RAGAS 0.4 requires a 'file_name' key in each document's metadata
# (used internally to track which document each chunk came from)
for document in docs:
    document.metadata["file_name"] = document.metadata["source"]

# Wrap the LangChain LLM and Embeddings in RAGAS-compatible wrappers
generator_llm = LangchainLLMWrapper(llm)
generator_embeddings = LangchainEmbeddingsWrapper(embeddings)

# Initialise the testset generator
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)

# default_query_distribution returns a weighted mix of synthesizers:
#   - SingleHopSpecificQuerySynthesizer  (simple factual questions)
#   - MultiHopAbstractQuerySynthesizer   (multi-doc reasoning)
#   - MultiHopSpecificQuerySynthesizer   (specific multi-doc questions)
query_distribution = default_query_distribution(generator_llm)

# ── Why run in a separate thread? ─────────────────────────────────────────────
# Jupyter's kernel already has a running asyncio event loop.
# RAGAS internally calls asyncio.run(), which conflicts with the existing loop.
# nest_asyncio patches this, but on Python 3.12+ it breaks sniffio's async-library
# detection → AsyncLibraryNotFoundError.
# Running RAGAS in a fresh thread gives it a clean event loop with no conflicts.
_result = {}

def _run_generation():
    try:
        _result["testset"] = generator.generate_with_langchain_docs(
            documents=docs,
            testset_size=10,
            query_distribution=query_distribution,
            raise_exceptions=False,  # Skip individual failed samples instead of crashing
        )
    except Exception as e:
        _result["error"] = e

thread = threading.Thread(target=_run_generation)
thread.start()
thread.join()  # Block until generation is complete

# Re-raise any exception that occurred inside the thread
if "error" in _result:
    raise _result["error"]

testset = _result["testset"]

# Preview the generated testset
testset.to_pandas()


C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\195214991.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(llm)
C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\195214991.py:14: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(embeddings)
Applying HeadlineSplitter: 100%|██████████| 3/3 [00:00<00:00, 808.46it/s]


ValueError: 'headlines' property not found in this node

In [59]:
# ── ADVANCED TESTSET GENERATION (optional, more control over transforms) ──────
# Use this cell instead of the simple generation above if you want to:
#   - Customise which document transforms are applied
#   - Skip transforms that crash (e.g. HeadlinesExtractor fails on short docs)
#   - Control exactly what relationships are built in the Knowledge Graph

import threading
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import default_query_distribution
from ragas.testset.graph import NodeType
from ragas.testset.transforms.extractors import EmbeddingExtractor, SummaryExtractor
from ragas.testset.transforms.extractors.llm_based import NERExtractor, ThemesExtractor
from ragas.testset.transforms.filters import CustomNodeFilter
from ragas.testset.transforms.relationship_builders import CosineSimilarityBuilder, OverlapScoreBuilder
from ragas.testset.transforms.engine import Parallel
from ragas.utils import num_tokens_from_string

# Wrap LangChain LLM and Embeddings in RAGAS-compatible wrappers
generator_llm = LangchainLLMWrapper(llm)
generator_embeddings = LangchainEmbeddingsWrapper(embeddings)

# Initialise the generator
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)

# Balanced mix of simple, reasoning, and multi-context questions
query_distribution = default_query_distribution(generator_llm)

# ── Custom Transform Pipeline ─────────────────────────────────────────────────
# Only process DOCUMENT nodes that have enough content (>100 tokens)
def filter_doc(node):
    return node.type == NodeType.DOCUMENT and num_tokens_from_string(node.properties["page_content"]) > 100

# Process all DOCUMENT nodes (used for theme extraction)
def filter_docs(node):
    return node.type == NodeType.DOCUMENT

# Step 1: Summarise each document node using the LLM
summary_extractor = SummaryExtractor(llm=generator_llm, filter_nodes=filter_doc)

# Step 2: Convert summaries to embeddings for similarity comparison
summary_emb_extractor = EmbeddingExtractor(
    embedding_model=generator_embeddings,
    property_name="summary_embedding",
    embed_property_name="summary",
    filter_nodes=filter_doc,
)

# Step 3: Build cosine-similarity edges between document nodes
cosine_sim_builder = CosineSimilarityBuilder(
    property_name="summary_embedding",
    new_property_name="summary_similarity",
    threshold=0.5,
    filter_nodes=filter_doc,
)

# Extract named entities from each node (used for overlap scoring)
ner_extractor = NERExtractor(llm=generator_llm)

# Build edges based on shared named entities between nodes
ner_overlap_sim = OverlapScoreBuilder(threshold=0.01)

# Extract high-level themes from each document node
theme_extractor = ThemesExtractor(llm=generator_llm, filter_nodes=filter_docs)

# Filter out low-quality nodes that are unlikely to generate good questions
node_filter = CustomNodeFilter(llm=generator_llm)

# Run transforms: sequential steps where needed, parallel where independent
custom_transforms = [
    summary_extractor,                                               # Must run first (summaries needed for embeddings)
    node_filter,                                                     # Filter nodes before heavy processing
    Parallel(summary_emb_extractor, theme_extractor, ner_extractor), # Independent — run in parallel
    Parallel(cosine_sim_builder, ner_overlap_sim),                   # Build relationships in parallel
]

# Run in a separate thread to avoid asyncio/nest_asyncio/sniffio conflicts in Jupyter
_adv_result = {}

def _run_advanced_generation():
    try:
        _adv_result["testset"] = generator.generate_with_langchain_docs(
            documents=docs,
            testset_size=8,
            query_distribution=query_distribution,
            transforms=custom_transforms,
            raise_exceptions=False,
        )
    except Exception as e:
        _adv_result["error"] = e

thread = threading.Thread(target=_run_advanced_generation)
thread.start()
thread.join()

if "error" in _adv_result:
    raise _adv_result["error"]

testset = _adv_result["testset"]


C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\1429513678.py:21: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(llm)
C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\1429513678.py:22: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(embeddings)
Applying ThemesExtractor:   0%|          | 0/3 [00:00<?, ?it/s]c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\euriai\langchain.py:323: RuntimeWarning: corout

In [60]:
testset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What are the ingridients and price of caprese ...,"[margherita pizza; $12; classic with tomato, m...","Caprese salad consists of fresh tomatoes, mozz...",Chef Amico,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,What role does Nero d’Avola play in Chef Amico...,"[In the heart of the old quarter of Palermo, a...","At Chef Amico, Nero d’Avola is featured promin...",Chef Amico,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
2,What makes Cannoli a standout dessert in Sicil...,"[In the charming streets of Palermo, tucked aw...",Cannoli is considered the crown jewel of Sicil...,Italian Cuisine Enthusiast,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
3,How did Amico's culinary journey and his exper...,[<1-hop>\n\nIn the heart of the old quarter of...,Amico's culinary journey was deeply influenced...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
4,What elements of Sicilian cuisine are reflecte...,[<1-hop>\n\nIn the heart of the old quarter of...,Chef Amico's restaurant embodies the essence o...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,How did Chef Amico's journey through Sicilian ...,[<1-hop>\n\nIn the heart of the old quarter of...,Chef Amico's journey through Sicilian cuisine ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,How did Chef Amico's upbringing in a Sicilian ...,[<1-hop>\n\nIn the charming streets of Palermo...,Chef Amico's upbringing in a Sicilian family s...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
7,What experiences in Italy shaped Chef Amico's ...,[<1-hop>\n\nIn the charming streets of Palermo...,Chef Amico's culinary journey was profoundly s...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
8,What experiences in Amico's childhood influenc...,[<1-hop>\n\nIn the charming streets of Palermo...,Amico's childhood experiences in his Nonna Luc...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [62]:
testset.to_pandas().shape

(9, 7)

In [63]:
# Build a Chroma vector store from the document chunks using Euriai embeddings.
# This is used by the retriever to find the most relevant context for each query.
vectorstore = Chroma.from_documents(chunks, embeddings)

# as_retriever() returns the top-k most similar chunks by default (k=4)
retriever = vectorstore.as_retriever()

print("Vector store built with", vectorstore._collection.count(), "chunks")


Vector store built with 33 chunks


In [64]:
# RAG prompt: instruct the LLM to answer ONLY from the retrieved context.
# This prevents hallucination by grounding the answer in the provided documents.
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = PromptTemplate(template=template, input_variables=["context", "question"])


In [65]:
# Build the RAG chain using LangChain Expression Language (LCEL):
#   1. Retriever fetches relevant document chunks for the question
#   2. RunnablePassthrough passes the original question through unchanged
#   3. Prompt formats context + question into a prompt string
#   4. llm generates an answer based on the prompt
#   5. StrOutputParser extracts the plain string from the LLM response
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [66]:
# Load manually curated Q&A pairs from CSV.
# These serve as the ground truth for evaluation:
#   'question'     — the user query
#   'ground_truth' — the ideal reference answer
df = pd.read_csv("./questions_answers/qa.csv", delimiter=";")
questions = df["question"].tolist()
ground_truth = df["ground_truth"].tolist()

print(f"Loaded {len(questions)} evaluation questions")


Loaded 30 evaluation questions


In [67]:
df.head()

,question,ground_truth
0,Where was Amico born?,Amico was born in the heart of the old quarter...
1,What was Amico's early culinary influence?,Amico was influenced by the cooking in his Non...
2,What skill did Amico learn from Palermo's mark...,Amico learned to select the freshest fish and ...
3,Where in Italy did Amico gain culinary experie...,Amico gained culinary experience across variou...
4,"What is ""Chef Amico"" restaurant known for?",Chef Amico is known for combining Sicilian and...


In [113]:
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

# For each question:
#   1. Run the RAG chain to get the LLM-generated answer
#   2. Retrieve the context documents used to generate that answer
#   3. Package everything into a SingleTurnSample for RAGAS evaluation
#
# RAGAS 0.4 field names:
#   user_input        — the question asked
#   response          — the LLM-generated answer
#   retrieved_contexts — list of context strings used (from retriever)
#   reference         — the ground truth answer (from CSV)

samples = []
for query, gt in zip(questions, ground_truth):
    answer = rag_chain.invoke(query)
    contexts = [doc.page_content for doc in retriever.invoke(query)]
    samples.append(
        SingleTurnSample(
            user_input=query,
            response=answer,
            retrieved_contexts=contexts,
            reference=gt,
        )
    )

# Wrap all samples into an EvaluationDataset for batch evaluation
dataset = EvaluationDataset(samples=samples)
print(f"Built evaluation dataset with {len(samples)} samples")


Built evaluation dataset with 30 samples


In [ ]:
# ── RAGAS 0.4 renamed the fields from the old 0.1.x API ──────────────────────
# Old name (0.1.x)  →  New name (0.4.x)
#   question        →  user_input
#   contexts        →  retrieved_contexts
#   answer          →  response
#   ground_truth    →  reference
#
# We rename them back here just for readability — the actual RAGAS internals
# always use the new names (user_input, retrieved_contexts, response, reference).

dataset.to_pandas().rename(columns={
    "user_input":          "question",
    "retrieved_contexts":  "contexts",
    "response":            "answer",
    "reference":           "ground_truth",
}).head()


,user_input,retrieved_contexts,response,reference
0,Where was Amico born?,[Amico's life was deeply entwined with the vib...,Amico was born in the heart of the old quarter...,Amico was born in the heart of the old quarter...
1,What was Amico's early culinary influence?,[Amico's life was deeply entwined with the vib...,Amico's early culinary influence was his Nonna...,Amico was influenced by the cooking in his Non...
2,What skill did Amico learn from Palermo's mark...,"[From a young age, Amico was immersed in the a...",Amico learned to choose the freshest fish from...,Amico learned to select the freshest fish and ...
3,Where in Italy did Amico gain culinary experie...,"[As he grew, so did his desire to explore beyo...",Amico gained culinary experience in various re...,Amico gained culinary experience across variou...
4,"What is ""Chef Amico"" restaurant known for?",[Chef Amico’s doors opened to a world where th...,Chef Amico restaurant is known for its warm an...,Chef Amico is known for combining Sicilian and...


In [115]:
from ragas import evaluate
# ── Correct import path for RAGAS 0.4 is ragas.metrics (NOT ragas.metrics.collections) ──
from ragas.metrics import (
    Faithfulness,                      # Is the answer grounded in the retrieved context? (no hallucinations)
    AnswerRelevancy,                   # Is the answer relevant and complete for the question?
    LLMContextRecall,                  # Did the retriever fetch all info needed to answer?
    LLMContextPrecisionWithReference,  # Are relevant chunks ranked higher than irrelevant ones?
    ContextRelevance,                  # Is the retrieved context relevant to the question?
)

# Wrap LLM and Embeddings in RAGAS-compatible wrappers for the evaluator
# - LLM is used to judge: Faithfulness, ContextRecall, ContextPrecision, ContextRelevance
# - Embeddings are used for: AnswerRelevancy (cosine similarity between questions)
eval_llm = LangchainLLMWrapper(llm)
eval_embeddings = LangchainEmbeddingsWrapper(embeddings)

metrics = [
    ContextRelevance(),
    LLMContextPrecisionWithReference(),
    LLMContextRecall(),
    Faithfulness(),
    AnswerRelevancy(),
]

print("Metrics configured:", [type(m).__name__ for m in metrics])


Metrics configured: ['ContextRelevance', 'LLMContextPrecisionWithReference', 'LLMContextRecall', 'Faithfulness', 'AnswerRelevancy']


C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\294396069.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\294396069.py:3: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\294396069.py:3: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\C90008809\AppData\Local\Temp\ipyker

In [ ]:
import asyncio
import threading
from ragas import aevaluate

_eval_result = {}

# ── Define the async evaluation coroutine ────────────────────────────────────
# All LLM/embedding/metric objects are created INSIDE this async function so
# that any async HTTP clients are bound to the SAME event loop that runs them.
async def _eval_async(eval_dataset):
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.metrics import (
        Faithfulness,
        AnswerRelevancy,
        LLMContextRecall,
        LLMContextPrecisionWithReference,
        ContextRelevance,
    )
    _llm  = LangchainLLMWrapper(llm)
    _emb  = LangchainEmbeddingsWrapper(embeddings)
    _metrics = [
        ContextRelevance(),
        LLMContextPrecisionWithReference(),
        LLMContextRecall(),
        Faithfulness(),
        AnswerRelevancy(),
    ]
    return await aevaluate(
        dataset=eval_dataset,
        metrics=_metrics,
        llm=_llm,
        embeddings=_emb,
        raise_exceptions=False,
    )

# ── Run in a fresh thread with its own event loop ────────────────────────────
# Python 3.12+ asyncio.wait_for() requires being inside a proper asyncio Task.
# A fresh thread + loop.run_until_complete() creates that Task context cleanly,
# with no interference from Jupyter's existing event loop.
def _run_in_thread(eval_dataset):
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        _eval_result["result"] = loop.run_until_complete(_eval_async(eval_dataset))
    except Exception as e:
        _eval_result["error"] = e
    finally:
        loop.close()

eval_thread = threading.Thread(target=_run_in_thread, args=(dataset,))
eval_thread.start()
eval_thread.join()

if "error" in _eval_result:
    raise _eval_result["error"]

result = _eval_result["result"]
print(result)


Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]


RuntimeError: Timeout should be used inside a task

In [111]:
result.to_pandas()

,user_input,retrieved_contexts,response,reference,nv_context_relevance,llm_context_precision_with_reference,context_recall,faithfulness,answer_relevancy
0,Where was Amico born?,[Amico's life was deeply entwined with the vib...,Amico was born in the heart of the old quarter...,Amico was born in the heart of the old quarter...,NaN,NaN,NaN,NaN,NaN
1,What was Amico's early culinary influence?,[Amico's life was deeply entwined with the vib...,Amico's early culinary influence was his Nonna...,Amico was influenced by the cooking in his Non...,NaN,NaN,NaN,NaN,NaN
2,What skill did Amico learn from Palermo's mark...,"[From a young age, Amico was immersed in the a...",Amico learned to choose the freshest fish from...,Amico learned to select the freshest fish and ...,NaN,NaN,NaN,NaN,NaN
3,Where in Italy did Amico gain culinary experie...,"[As he grew, so did his desire to explore beyo...",Amico gained culinary experience throughout It...,Amico gained culinary experience across variou...,NaN,NaN,NaN,NaN,NaN
4,"What is ""Chef Amico"" restaurant known for?",[Chef Amico’s doors opened to a world where th...,Chef Amico restaurant is known for its warm an...,Chef Amico is known for combining Sicilian and...,NaN,NaN,NaN,NaN,NaN
5,What does Amico's restaurant menu reflect?,"[At Chef Amico, every dish told a story. The m...",Amico's restaurant menu reflects a tapestry of...,The menu reflects Amico's culinary journey and...,NaN,NaN,NaN,NaN,NaN
6,How does Amico perceive hospitality?,"[For Amico, hospitality was an art form. He be...",Amico perceives hospitality as an art form and...,Amico sees hospitality as an art of celebratin...,NaN,NaN,NaN,NaN,NaN
7,"What distinguishes ""Chef Amico"" in Palermo?","[Returning to Palermo with a vision, Amico ope...","""Chef Amico"" is distinguished in Palermo by it...",Chef Amico is distinguished by Amico's dedicat...,NaN,NaN,NaN,NaN,NaN
8,What activities does Amico engage in besides c...,[Amico's life was deeply entwined with the vib...,"Besides cooking, Amico engages in mentoring yo...","Amico mentors young chefs, conducts culinary w...",NaN,NaN,NaN,NaN,NaN
9,How is Amico's legacy beyond his dishes?,[Amico’s legacy is not just in the dishes he c...,Amico's legacy extends beyond his dishes throu...,Amico's legacy lies in his community involveme...,NaN,NaN,NaN,NaN,NaN


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Convert RAGAS results to a DataFrame
df_results = result.to_pandas()

print("Available columns:", df_results.columns.tolist())

# ── Dynamically detect metric columns ────────────────────────────────────────
# RAGAS 0.4 metric names can differ from the class name (e.g. ContextRelevance
# has internal name 'nv_context_relevance', LLMContextRecall → 'context_recall').
# We detect them automatically by excluding the known input/output columns.
non_metric_cols = {"user_input", "response", "retrieved_contexts", "reference"}
metric_cols = [
    c for c in df_results.columns
    if c not in non_metric_cols and pd.api.types.is_numeric_dtype(df_results[c])
]

print("Metric columns found:", metric_cols)

if not metric_cols:
    print("No metric columns found — all scores may still be NaN. Check LLM errors above.")
else:
    heatmap_data = df_results[metric_cols]

    # Red → Green colormap: red = poor score (0.0), green = good score (1.0)
    cmap = LinearSegmentedColormap.from_list("red_green", ["red", "yellow", "green"])

    fig, ax = plt.subplots(figsize=(max(8, len(metric_cols) * 2), max(6, len(df_results) * 0.5)))
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        cmap=cmap,
        vmin=0,
        vmax=1,
        ax=ax,
    )

    # Use shortened question labels on Y-axis (RAGAS 0.4 field: 'user_input')
    labels = [q[:60] + "…" if len(q) > 60 else q for q in df_results["user_input"]]
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, rotation=0, fontsize=8)
    ax.set_title("RAGAS Evaluation Results")
    plt.tight_layout()
    plt.show()


In [127]:
import asyncio
import threading
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas import aevaluate

# ── Single-sample spot-check ──────────────────────────────────────────────────
# Build one sample using the RAGAS 0.4 field names.
# Mapping from old (0.1.x) → new (0.4.x):
#   question     → user_input
#   contexts     → retrieved_contexts
#   answer       → response
#   ground_truth → reference

single_sample = SingleTurnSample(
    user_input="What is the capital of France?",           # question
    retrieved_contexts=["Paris is the capital and most populous city of France."],  # contexts
    response="The capital of France is Paris.",            # answer
    reference="Paris",                                     # ground_truth
)

single_dataset = EvaluationDataset(samples=[single_sample])

# Preview with familiar column names
single_dataset.to_pandas().rename(columns={
    "user_input":         "question",
    "retrieved_contexts": "contexts",
    "response":           "answer",
    "reference":          "ground_truth",
})


,question,contexts,answer,ground_truth
0,What is the capital of France?,[Paris is the capital and most populous city o...,The capital of France is Paris.,Paris


In [128]:
import asyncio
import threading
from ragas import aevaluate

# ── Evaluate the single sample ────────────────────────────────────────────────
_single_eval = {}

async def _eval_single_async():
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.metrics import (
        Faithfulness,
        AnswerRelevancy,
        LLMContextRecall,
        LLMContextPrecisionWithReference,
        ContextRelevance,
    )
    return await aevaluate(
        dataset=single_dataset,
        metrics=[
            ContextRelevance(),
            LLMContextPrecisionWithReference(),
            LLMContextRecall(),
            Faithfulness(),
            AnswerRelevancy(),
        ],
        llm=LangchainLLMWrapper(llm),
        embeddings=LangchainEmbeddingsWrapper(embeddings),
        raise_exceptions=False,
    )

def _run_single():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        _single_eval["result"] = loop.run_until_complete(_eval_single_async())
    except Exception as e:
        _single_eval["error"] = e
    finally:
        loop.close()

t = threading.Thread(target=_run_single)
t.start()
t.join()

if "error" in _single_eval:
    raise _single_eval["error"]

single_result = _single_eval["result"]

# Show results with familiar column names
single_result.to_pandas().rename(columns={
    "user_input":         "question",
    "retrieved_contexts": "contexts",
    "response":           "answer",
    "reference":          "ground_truth",
})


C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\2864525457.py:11: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\2864525457.py:11: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\2864525457.py:11: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\C90008809\AppData\Local\Temp\

,question,contexts,answer,ground_truth,nv_context_relevance,llm_context_precision_with_reference,context_recall,faithfulness,answer_relevancy
0,What is the capital of France?,[Paris is the capital and most populous city o...,The capital of France is Paris.,Paris,NaN,NaN,NaN,NaN,NaN


In [151]:
# =============================
# Imports
# =============================
from euriai.langchain import create_chat_model
from euriai.langchain import EuriaiEmbeddings

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# ⚠️ CRITICAL: import from langchain_core, not langchain.chains
from langchain_core.output_parsers import StrOutputParser

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\1126967778.py:19: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\1126967778.py:19: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\1126967778.py:19: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. E

In [159]:
# =====================================================
# 1. Documents
# =====================================================
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

documents = [
    Document(page_content="Retrieval Augmented Generation combines retrieval with LLMs."),
    Document(page_content="RAG improves factual accuracy using external documents."),
    Document(page_content="LangChain is a framework for building LLM applications."),
]

# =====================================================
# 2. Split Docs
# =====================================================
splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40)
docs = splitter.split_documents(documents)

# =====================================================
# 3. Vector Store
# =====================================================
embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# =====================================================
# 4. LLM
# =====================================================

#llm = create_chat_model(api_key=api_key, model="gpt-4o-mini", temperature=0.1)
llm = ChatOpenAI(
    model="stepfun/step-3.5-flash:free",
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
)

# =====================================================
# 5. Prompt
# =====================================================
prompt = ChatPromptTemplate.from_template(
    """Answer the question using the following context only.

Context:
{context}

Question:
{question}
"""
)

# =====================================================
# 6. Chain (FIXED)
# =====================================================
rag_chain = (
    {
        "docs": retriever,
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(lambda x: {
        "context": format_docs(x["docs"]),
        "question": x["question"],
        "docs": x["docs"],   # keep raw docs
    })
    | RunnableLambda(lambda x: {
        "answer": llm.invoke(prompt.format(**x)).content,
        "contexts": [doc.page_content for doc in x["docs"]],
    })
)

# =====================================================
# 7. Run RAG
# =====================================================
question = "What is Retrieval Augmented Generation?"
response = rag_chain.invoke(question)

answer = response["answer"]
contexts = response["contexts"]

# =====================================================
# 8. RAGAS Dataset
# =====================================================
dataset = Dataset.from_dict({
    "question": [question],
    "answer": [answer],
    "contexts": [contexts],  # list of list
    "ground_truth": [
        "Retrieval Augmented Generation improves LLM responses by retrieving relevant documents."
    ],
})

# =====================================================
# 9. RAGAS Evaluation
# =====================================================

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

scores = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],llm=ragas_llm, embeddings=ragas_embeddings, raise_exceptions=True
)

print("Answer:\n", answer)

print("\nContexts:")
for c in contexts:
    print("-", c)

print("\nRAGAS Scores:\n", scores)

C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\691531207.py:102: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
C:\Users\C90008809\AppData\Local\Temp\ipykernel_9972\691531207.py:103: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)
Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]


RuntimeError: Timeout should be used inside a task

In [157]:
[contexts]

[['Retrieval Augmented Generation combines retrieval with LLMs.',
  'RAG improves factual accuracy using external documents.']]

In [1]:
print(1)

1


In [1]:
# =============================
# Imports
# =============================
from euriai.langchain import create_chat_model
from euriai.langchain import EuriaiEmbeddings

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# ⚠️ CRITICAL: import from langchain_core, not langchain.chains
from langchain_core.output_parsers import StrOutputParser

from ragas import evaluate, aevaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

import nest_asyncio
import asyncio

# =====================================================
# 1. Documents
# =====================================================
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

documents = [
    Document(page_content="Retrieval Augmented Generation combines retrieval with LLMs."),
    Document(page_content="RAG improves factual accuracy using external documents."),
    Document(page_content="LangChain is a framework for building LLM applications."),
]

# =====================================================
# 2. Split Docs
# =====================================================
splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40)
docs = splitter.split_documents(documents)

# =====================================================
# 3. Vector Store
# =====================================================
embeddings = EuriaiEmbeddings(
    api_key="euri-49e67794160469861d51db4b89da0c40be0457fd1303002d1fbf682ad44c512e",
    model="text-embedding-3-small"
)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# =====================================================
# 4. LLM
# =====================================================

#llm = create_chat_model(api_key=api_key, model="gpt-4o-mini", temperature=0.1)
llm = ChatOpenAI(
    model="stepfun/step-3.5-flash:free",
    base_url="https://openrouter.ai/api/v1",
    api_key="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",
)

print(llm.invoke("What is the capital of France?").content)

# =====================================================
# 5. Prompt
# =====================================================
prompt = ChatPromptTemplate.from_template(
    """Answer the question using the following context only.

Context:
{context}

Question:
{question}
"""
)

# =====================================================
# 6. Chain (FIXED)
# =====================================================
rag_chain = (
    {
        "docs": retriever,
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(lambda x: {
        "context": format_docs(x["docs"]),
        "question": x["question"],
        "docs": x["docs"],   # keep raw docs
    })
    | RunnableLambda(lambda x: {
        "answer": llm.invoke(prompt.format(**x)).content,
        "contexts": [doc.page_content for doc in x["docs"]],
    })
)

# =====================================================
# 7. Run RAG
# =====================================================
question = "What is Retrieval Augmented Generation?"
response = rag_chain.invoke(question)

answer = response["answer"]
contexts = response["contexts"]

print(contexts)
print(answer)

# =====================================================
# 8. RAGAS Dataset
# =====================================================
dataset = Dataset.from_dict({
    "question": [question],
    "answer": [answer],
    "contexts": [contexts],  # list of list
    "ground_truth": [
        "Retrieval Augmented Generation improves LLM responses by retrieving relevant documents."
    ],
})

# =====================================================
# 9. RAGAS Evaluation
# =====================================================

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
import asyncio

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# Apply nest_asyncio to allow nested loops in Jupyter
nest_asyncio.apply()

scores = await aevaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],llm=ragas_llm, embeddings=ragas_embeddings, raise_exceptions=True)

print("Answer:\n", answer)

print("\nContexts:")
for c in contexts:
    print("-", c)

print("\nRAGAS Scores:\n", scores)

c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\C90008809\AppData\Local\Temp\ipykernel_28368\2522151943.py:19: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\C90008809\AppData\Local\Temp\ipykernel_28368\2522151943.py:19: D

The capital of France is **Paris**.


C:\Users\C90008809\AppData\Local\Temp\ipykernel_28368\2522151943.py:132: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
C:\Users\C90008809\AppData\Local\Temp\ipykernel_28368\2522151943.py:133: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)
C:\Users\C90008809\AppData\Local\Temp\ipykernel_28368\2522151943.py:138: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator i

['Retrieval Augmented Generation combines retrieval with LLMs.', 'RAG improves factual accuracy using external documents.']
Retrieval Augmented Generation (RAG) is a method that combines retrieval with large language models (LLMs) to improve factual accuracy by utilizing external documents.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]


RuntimeError: Timeout should be used inside a task